# Chapter 14: Wiener Processes and Itô's Lemma
## Simulating Continuous-Time Stochastic Processes with Python

In this notebook, we explore the mathematical foundations of continuous-time stochastic processes as presented in **Chapter 14** of John Hull's *Options, Futures, and Other Derivatives*.

Because this chapter is heavily theoretical and mathematical, we will focus on **bringing the equations to life using Python simulations and rich visualizations**. We will cover:
1. **The Markov Property**: Understanding memoryless processes.
2. **Continuous-Time Stochastic Processes**: Simulating Standard and Generalized Wiener Processes.
3. **Itô Processes**: Processes with time-varying and state-varying drifts.
4. **The Process for a Stock Price (Geometric Brownian Motion)**: The foundational model for Black-Scholes, mapping equations to multiple simulated stock price paths.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:,.4f}'.format
np.random.seed(42) # For reproducible simulations

# Theme colors
navy_blue = '#1E3A8A'
emerald = '#10B981'
crimson = '#EF4444'
amber = '#F59E0B'
violet = '#8B5CF6'
dark_gray = '#374151'


---
## 14.1 The Markov Property

A **Markov process** is a particular type of stochastic process where only the current value of a variable is relevant for predicting the future. The past history is irrelevant.

### Summary
* **Weak Form of Market Efficiency**: The Markov property of stock prices is consistent with the weak form of market efficiency, meaning past prices are already priced into the current value.
* **Variance Additivity**: For Markov processes, the *variances* of changes in successive time periods are additive, but standard deviations are not. Consequently, uncertainty (standard deviation) increases with the **square root of time** ($\sqrt{t}$).


### Intuitive Analogy: The "Amnesiac" Process
> **The Rule**: The future depends only on *where you are right now*, not on *how you got there*. The process has no memory.
> * **Real-world example**: A game of Monopoly or Roulette. If your token is on "Boardwalk", your probability of moving to the next specific square depends only on your current position and the dice roll. The path you took to arrive at Boardwalk is entirely forgotten and irrelevant.
> * **In Finance**: If a stock is $100 today, the probability of it moving to $101 tomorrow depends only on the fact that it is $100 today. It doesn't matter if it rallied from $50 or crashed from $200 to get there.


---
## 14.2 Continuous-Time Stochastic Processes

### 1. The Wiener Process (Brownian Motion)
A variable $z$ follows a standard Wiener process if it has two properties:
1. The change $\Delta z$ during a small period $\Delta t$ is $\Delta z = \epsilon \sqrt{\Delta t}$, where $\epsilon \sim \mathcal{N}(0, 1)$.
2. Values of $\Delta z$ for any two different short intervals are independent.

### Example: Simulating a Wiener Process
Let's simulate a Wiener Process over 1 year ($T=1$) with $N=500$ steps.


### Intuitive Analogy: The "Drunkard in the Park"
> **The Rule**: It is a specific type of Markov process where movement is continuous, smooth, and **purely random**. It has no trend, and steps are unpredictable.
> * **Real-world example**: A speck of dust floating in a glass of water, being randomly bumped by water molecules from all directions. It wanders aimlessly, without any preference for going left or right. Another analogy is a drunkard taking random steps in an open field. After an hour, he will be somewhere far from the start, but the exact direction is pure chance. The uncertainty (distance traveled) grows with the square root of time ($\sqrt{t}$).


In [ ]:
T = 1.0
N = 500
dt = T / N
t = np.linspace(0, T, N + 1)

# Generate epsilon: standard normal random variables
epsilon = np.random.standard_normal(N)

# Calculate dz = epsilon * sqrt(dt)
dz = epsilon * np.sqrt(dt)

# Accumulate dz to build the path z(t)
z = np.zeros(N + 1)
z[1:] = np.cumsum(dz)

plt.figure(figsize=(10, 6))
plt.plot(t, z, color=navy_blue, lw=1.5)
plt.title('Simulation of a Standard Wiener Process ($dz$)', fontsize=14)
plt.xlabel('Time (Years)', fontsize=12)
plt.ylabel('Value of z', fontsize=12)
plt.grid(alpha=0.3)
plt.axhline(0, color='black', lw=1, linestyle='--')
plt.show()


### 2. Generalized Wiener Process
A generalized Wiener process for a variable $x$ incorporates a constant expected drift rate ($a$) and a constant variance rate ($b^2$):
$$dx = a \, dt + b \, dz$$

Which in discrete time translates to:
$$\Delta x = a \Delta t + b \epsilon \sqrt{\Delta t}$$


### Intuitive Analogy: The "Drunkard on a Moving Walkway"
> **The Rule**: A standard Wiener process plus a constant trend.
> * **Real-world example**: Imagine the same drunkard from before, taking random, aimless steps. But now, he is on an airport moving walkway. He still wanders randomly (constant variance/wobble), but the walkway is continuously pushing him forward at a constant speed (constant drift/trend).


In [ ]:
# Example 14.2 parameters
a = 20    # Drift rate (expected change per year)
b = 30    # standard deviation per year (variance is 900)
x0 = 50   # Initial cash position

dx = a * dt + b * dz
x = np.zeros(N + 1)
x[0] = x0
x[1:] = x0 + np.cumsum(dx)
expected_drift = x0 + a * t

plt.figure(figsize=(10, 6))
plt.plot(t, x, color=crimson, lw=1.5, label='Generalized Wiener Process Path')
plt.plot(t, expected_drift, color=dark_gray, lw=2, linestyle='--', label='Expected Drift ($a \, dt$)')
plt.title('Generalized Wiener Process: $dx = 20dt + 30dz$', fontsize=14)
plt.xlabel('Time (Years)', fontsize=12)
plt.ylabel('Cash Position', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### 3. Itô Process
An Itô process is a generalized Wiener process where the parameters $a$ and $b$ are not constants, but rather functions of the underlying variable $x$ and time $t$:
$$dx = a(x, t) dt + b(x, t) dz$$


### Intuitive Analogy: The "Leaf in a Wild River"
> **The Rule**: It is a Wiener process with a trend, **BUT** both the strength of the trend (speed) and the strength of the randomness (turbulence) **change** depending on *where* you are and *when* you are there.
> * **Real-world example**: A leaf floating down a river flowing down a mountain. The river's current (the drift) pushes the leaf, but the current is not constant—it is fast in the middle and slow near the edges. Also, the turbulence (the volatility) changes—it is calm in deep waters but bubbles violently near shallow rocks. To predict the leaf's next move, you need to know the exact conditions of the river at that exact spot and time.
> * **In Finance**: This is where Black-Scholes lives! A stock price is an Itô process because a company's volatility often changes as its price changes. If a stock crashes from $100 to $10, the company nears bankruptcy (approaching the "rocks in the river"), and its volatility (turbulence) explodes. The rules of the game change as the game is played.



### Summary: The Evolution from Markov to Itô
To understand how we model a stock price, it is extremely helpful to see how these mathematical concepts build upon each other like Russian nesting dolls:

1. **Markov Process (The Foundation)**: A broad category of stochastic processes where only the present matters, not the past. (It can be discrete or continuous).
2. **$\to$ Wiener Process ($dz$)**: We take a Markov process and restrict it: it must be continuous over time, have a mean of zero (no trend), and a variance of 1.0 per year. It is **pure, standardized randomness**.
3. **$\to$ Generalized Wiener Process ($dx = a dt + b dz$)**: We take the pure Wiener process and give it a **constant trend** ($a$) and a **constant scale for the randomness** ($b$). It's no longer just zero-mean; it drifts steadily and wobbles consistently.
4. **$\to$ Itô Process ($dx = a(x,t) dt + b(x,t) dz$)**: We realize the real world isn't constant. We take the Generalized Wiener process and allow the trend ($a$) and the randomness ($b$) to **change dynamically** depending on our current state ($x$) and time ($t$).
5. **$\to$ Geometric Brownian Motion ($dS = \mu S dt + \sigma S dz$)**: Finally, we apply the Itô process to stocks. We realize that a stock's expected return and volatility scale proportionally with the stock price itself ($S$). This gives us the exact continuous-time mathematical model used to price options in the real world!


---
## 14.3 The Process for a Stock Price

A simple generalized Wiener process is inappropriate for a stock price because the required return from a stock should be independent of its price (the drift should be proportional to the price, not a constant flat amount).

The widely used model for stock price behavior is **Geometric Brownian Motion**:
$$dS = \mu S dt + \sigma S dz$$

Dividing by $S$ gives the continuous return:
$$\frac{dS}{S} = \mu dt + \sigma dz$$

### Discrete-Time Model
In discrete time $\Delta t$, the return is:
$$\frac{\Delta S}{S} = \mu \Delta t + \sigma \epsilon \sqrt{\Delta t}$$

### Simulation Example
Let's simulate **multiple paths** for a stock price to visualize the distribution of terminal prices. 
* $S_0 = 100$
* Expected return $\mu = 0.15$ (15% per year)
* Volatility $\sigma = 0.30$ (30% per year)
* 1 year simulation ($T=1$), generating 20 different possible market paths.


In [ ]:
S0 = 100
mu = 0.15
sigma = 0.30
T = 1.0
N = 252 # Trading days in a year
dt = T / N
M = 20  # Number of paths to simulate

t_days = np.linspace(0, T, N + 1)
S_paths = np.zeros((N + 1, M))
S_paths[0] = S0

# Simulate M paths
for i in range(1, N + 1):
    epsilon = np.random.standard_normal(M)
    # Discrete time step: S(t) = S(t-1) + mu * S(t-1) * dt + sigma * S(t-1) * epsilon * sqrt(dt)
    dS = mu * S_paths[i-1] * dt + sigma * S_paths[i-1] * epsilon * np.sqrt(dt)
    S_paths[i] = S_paths[i-1] + dS

# Plotting the paths
plt.figure(figsize=(12, 7))
for m in range(M):
    plt.plot(t_days, S_paths[:, m], lw=1.5, alpha=0.7)

# Expected deterministic path S0 * exp(mu * t)
expected_S = S0 * np.exp(mu * t_days)
plt.plot(t_days, expected_S, color='black', lw=3, linestyle='--', label='Expected Value (No Uncertainty)')

plt.title(f'Geometric Brownian Motion - {M} Simulated Stock Price Paths', fontsize=15)
plt.xlabel('Time (Years)', fontsize=12)
plt.ylabel('Stock Price ($S_t$)', fontsize=12)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


### Summary of Key Mathematical Takeaways:
1. **$dz$ dictates the randomness**: It generates normally distributed noise scaled by $\sqrt{dt}$.
2. **Drift vs Volatility**: Over short periods $\Delta t$, the random term $\sigma \epsilon \sqrt{\Delta t}$ dominates the expected drift term $\mu \Delta t$, causing the jaggedness seen in the plots.
3. **Lognormal Property**: Because $\frac{\Delta S}{S}$ is normally distributed, the final stock price $S_T$ ends up having a **lognormal distribution** (prices cannot drop below zero, but can rise infinitely).


---
## 14.4 Itô's Lemma

Itô's Lemma is arguably the most important result in quantitative finance. If a variable $x$ follows an Itô process:
$$dx = a(x,t)dt + b(x,t)dz$$

Then, any function $G(x,t)$ of $x$ and $t$ also follows an Itô process. Itô's Lemma gives us the exact formula for $dG$:
$$dG = \left( \frac{\partial G}{\partial x} a + \frac{\partial G}{\partial t} + \frac{1}{2} \frac{\partial^2 G}{\partial x^2} b^2 \right) dt + \frac{\partial G}{\partial x} b \, dz$$

### Intuitive Analogy: The "Bumpy Road" and the Suspension
> **The Rule**: In standard calculus, we ignore $dx^2$ because it is too small to matter. But in stochastic calculus, $dz^2 = dt$. The randomness accumulates over time. Therefore, we **must** keep the second derivative term (the $\frac{1}{2} \frac{\partial^2 G}{\partial x^2} b^2$ term).
> * **Real-world example**: Imagine driving a car up a hill. In standard calculus (a perfectly smooth road), to calculate your altitude gain, you only care about your speed and the slope of the hill (the first derivative). However, if you are driving on an extremely **bumpy, rocky road** (a stochastic path), the intense vibrations bounce your car up and down. Because of the curvature of your car's suspension (the second derivative), these massive vibrations actually change your average altitude over time! You cannot ignore the bumps; they have a real, cumulative effect.
> * **In Finance**: The "bumps" are the stock's volatility ($\sigma$). Because derivatives (like Call options) have curved payoffs (convexity), the stock's volatility actively increases the expected value of the option. Itô's Lemma is the mathematical tool that captures the exact value of these "bumps".

### Example: Applying Itô's Lemma to a Stock's Return
Let a stock follow Geometric Brownian Motion: $dS = \mu S dt + \sigma S dz$.
We want to find the process followed by the log return: $G = \ln(S)$.

1. $\frac{\partial G}{\partial S} = \frac{1}{S}$
2. $\frac{\partial^2 G}{\partial S^2} = -\frac{1}{S^2}$
3. $\frac{\partial G}{\partial t} = 0$

Plugging this into Itô's Lemma:
$$dG = \left( \frac{1}{S} \mu S + 0 + \frac{1}{2} \left( -\frac{1}{S^2} \right) \sigma^2 S^2 \right) dt + \frac{1}{S} \sigma S dz$$
$$d(\ln S) = \left( \mu - \frac{\sigma^2}{2} \right) dt + \sigma dz$$

**Conclusion**: The continuously compounded return of a stock is normally distributed with a mean of $(\mu - \frac{\sigma^2}{2})$ and standard deviation $\sigma \sqrt{t}$. Notice the variance drag ($- \frac{\sigma^2}{2}$) caused by the "bumpy road"!


In [ ]:
# Simulating Itô's Lemma on log returns
S0_ito = 100
mu_ito = 0.15
sigma_ito = 0.30
T_ito = 1.0
N_ito = 252
dt_ito = T_ito / N_ito
M_ito = 10000 # Let's simulate 10,000 paths to prove the distribution

# Generate the random dz for all paths at once
epsilon_ito = np.random.standard_normal((N_ito, M_ito))
dz_ito = epsilon_ito * np.sqrt(dt_ito)

# Using Itô's Lemma directly to simulate the log return:
# d(ln S) = (mu - 0.5 * sigma^2) * dt + sigma * dz
d_lnS = (mu_ito - 0.5 * sigma_ito**2) * dt_ito + sigma_ito * dz_ito

# Accumulate returns
lnS_paths = np.zeros((N_ito + 1, M_ito))
lnS_paths[0] = np.log(S0_ito)
lnS_paths[1:] = np.log(S0_ito) + np.cumsum(d_lnS, axis=0)

# Convert back to price
S_paths_ito = np.exp(lnS_paths)

# Get final log returns
final_log_returns = lnS_paths[-1] - np.log(S0_ito)

# Calculate theoretical vs simulated mean and variance
theoretical_mean = (mu_ito - 0.5 * sigma_ito**2) * T_ito
theoretical_variance = (sigma_ito**2) * T_ito

simulated_mean = np.mean(final_log_returns)
simulated_variance = np.var(final_log_returns)

print("--- Itô's Lemma: Log Return Distribution Over 1 Year ---")
print(f"Theoretical Mean (mu - sigma^2 / 2): {theoretical_mean:.4f}")
print(f"Simulated Mean:                      {simulated_mean:.4f}\n")
print(f"Theoretical Variance (sigma^2):      {theoretical_variance:.4f}")
print(f"Simulated Variance:                  {simulated_variance:.4f}")

# Plotting the distribution of the final log returns
plt.figure(figsize=(10, 6))
plt.hist(final_log_returns, bins=50, density=True, color=amber, alpha=0.7, edgecolor='black')
plt.axvline(theoretical_mean, color='black', linestyle='--', lw=2, label=f'Mean = {theoretical_mean:.4f}')
plt.title("Distribution of Continuously Compounded Returns $\\ln(S_T/S_0)$", fontsize=14)
plt.xlabel("Return", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.show()


---
### Example: Applying Itô's Lemma to Forward Contracts
Another critical application of Itô's Lemma is finding the stochastic process followed by the forward price of a stock. Let $F$ be the forward price at time $t$ for a contract maturing at $T$. For a non-dividend-paying stock, we know:
$$F = S e^{r(T-t)}$$

We assume the underlying stock follows Geometric Brownian Motion: $dS = \mu S dt + \sigma S dz$.
Let's apply Itô's Lemma where our function is $G(S, t) = F = S e^{r(T-t)}$:

1. $\frac{\partial F}{\partial S} = e^{r(T-t)}$
2. $\frac{\partial^2 F}{\partial S^2} = 0$ (Because $F$ is a linear function of $S$, the "bumpy road" curvature is zero!)
3. $\frac{\partial F}{\partial t} = -r S e^{r(T-t)} = -r F$

Plugging these derivatives into Itô's formula:
$$dF = \left( e^{r(T-t)} \mu S - r F + 0 \right) dt + e^{r(T-t)} \sigma S dz$$

Since $e^{r(T-t)} S = F$, we can simplify this to:
$$dF = (\mu - r) F dt + \sigma F dz$$

**Conclusion**: The forward price also follows Geometric Brownian Motion! Its volatility $\sigma$ is exactly the same as the stock's volatility. However, its expected growth rate is $(\mu - r)$. 

> **Important Insight**: In a *risk-neutral world*, the expected return of the stock is the risk-free rate ($\mu = r$). If we plug this in, the drift term disappears entirely, leaving $dF = \sigma F dz$. This proves mathematically that **in a risk-neutral world, the forward price has an expected drift of zero** (it is a martingale).


In [ ]:
# Simulating Stock Price vs Forward Price paths
S0_fwd = 100
r_fwd = 0.05
mu_fwd = 0.12  # Real-world drift
sigma_fwd = 0.20
T_fwd = 1.0
N_fwd = 252
dt_fwd = T_fwd / N_fwd

t_fwd_arr = np.linspace(0, T_fwd, N_fwd + 1)

# Generate one specific random path (dz) for the asset
np.random.seed(101)
epsilon_fwd = np.random.standard_normal(N_fwd)
dz_fwd = epsilon_fwd * np.sqrt(dt_fwd)

# Simulate Stock Price (S)
S_path = np.zeros(N_fwd + 1)
S_path[0] = S0_fwd
for i in range(1, N_fwd + 1):
    S_path[i] = S_path[i-1] + mu_fwd * S_path[i-1] * dt_fwd + sigma_fwd * S_path[i-1] * dz_fwd[i-1]

# Simulate Forward Price (F) directly using Itô's deduced process
# F0 = S0 * exp(r * T)
F_path = np.zeros(N_fwd + 1)
F_path[0] = S0_fwd * np.exp(r_fwd * T_fwd)
for i in range(1, N_fwd + 1):
    # dF = (mu - r) * F * dt + sigma * F * dz
    F_path[i] = F_path[i-1] + (mu_fwd - r_fwd) * F_path[i-1] * dt_fwd + sigma_fwd * F_path[i-1] * dz_fwd[i-1]

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(t_fwd_arr, S_path, color=navy_blue, lw=2, label='Stock Price Path ($S_t$)')
plt.plot(t_fwd_arr, F_path, color=crimson, lw=2, linestyle='--', label='Forward Price Path ($F_t$)')

# The deterministic convergence
plt.title('Stock Price vs Forward Price Over Time (Real-World $\\mu > r$)', fontsize=14)
plt.xlabel('Time (Years)', fontsize=12)
plt.ylabel('Price', fontsize=12)
plt.grid(alpha=0.3)
plt.legend()

# Highlight that as t approaches T, F converges to S
plt.text(0.8, S_path[-1] + 2, 'Convergence at $T$', fontsize=12, color=dark_gray, fontweight='bold')
plt.show()

print("Notice how the Forward Price starts higher due to the cost of carry $e^{rT}$, ")
print("shares the exact same volatility bumps ($\sigma$) as the stock throughout the path, ")
print("and strictly converges to the Stock Price exactly at expiration $T=1.0$!")


---
## 14.5 Fractional Brownian Motion (fBm)

While standard Brownian Motion (Wiener process) assumes that all future movements are entirely independent of the past, **Fractional Brownian Motion (fBm)** challenges this assumption. It introduces a parameter called the **Hurst index** ($H$), where $0 < H < 1$, which determines the "memory" or correlation of the process.

* **$H = 0.5$**: Standard Brownian Motion. The increments are completely independent. No memory.
* **$H > 0.5$ (Persistent)**: The increments are positively correlated. If the process went up in the past, it is more likely to keep going up. The path exhibits long-term memory and appears much "smoother" and trend-following.
* **$H < 0.5$ (Anti-persistent)**: The increments are negatively correlated. If the process went up, it is more likely to reverse and go down next. The path exhibits extreme mean-reversion and appears highly "rough" or jagged.

The covariance between two points in time $t$ and $s$ for an fBm process $B_H(t)$ is defined mathematically as:
$$E[B_H(t) B_H(s)] = \frac{1}{2} \left( t^{2H} + s^{2H} - |t-s|^{2H} \right)$$

### Intuitive Analogy: The Travelers
> **The Rule**: In standard Brownian motion, the traveler throws a coin at every step and has amnesia. In fBm, the traveler remembers their past steps and has a personality!
> * **$H > 0.5$ (The Stubborn Traveler)**: This traveler hates changing directions. If they took 5 steps north, they feel an immense psychological momentum to keep walking north. They create long, sweeping, smooth paths. (Momentum/Trend-following markets).
> * **$H < 0.5$ (The Rebellious Traveler)**: This traveler hates doing the same thing twice. If they step forward, their immediate reaction is to step backward just to be contrarian. They bounce wildly back and forth around the same area, creating incredibly rough and zigzagged paths. (Mean-reverting markets).


In [ ]:
# Simulating Fractional Brownian Motion (fBm) using Cholesky Decomposition
# Note: For large N, Cholesky can be slow, but for N=500 it computes instantly.

def simulate_fbm(H, N, T):
    t = np.linspace(0, T, N)
    # Construct the covariance matrix
    # cov(t, s) = 0.5 * (t^(2H) + s^(2H) - |t-s|^(2H))
    T_mat, S_mat = np.meshgrid(t, t)
    cov_matrix = 0.5 * (T_mat**(2*H) + S_mat**(2*H) - np.abs(T_mat - S_mat)**(2*H))
    
    # Add a tiny value to the diagonal to ensure numerical stability (positive semi-definite)
    cov_matrix += np.eye(N) * 1e-8
    
    # Cholesky decomposition: L * L.T = Covariance Matrix
    L = np.linalg.cholesky(cov_matrix)
    
    # Multiply L by standard normal variables to get the correlated path
    np.random.seed(42) # Use the same seed to compare the EXACT same random shocks
    Z = np.random.standard_normal(N)
    fbm_path = np.dot(L, Z)
    return t, fbm_path

N_fbm = 500
T_fbm = 1.0

# Simulate for Anti-persistent, Standard, and Persistent
t_fbm, path_rough = simulate_fbm(0.2, N_fbm, T_fbm)   # H = 0.2 (Rough)
t_fbm, path_standard = simulate_fbm(0.5, N_fbm, T_fbm) # H = 0.5 (Standard)
t_fbm, path_smooth = simulate_fbm(0.8, N_fbm, T_fbm)   # H = 0.8 (Smooth)

# Plotting
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

axes[0].plot(t_fbm, path_rough, color=crimson, lw=1.5)
axes[0].set_title('Anti-Persistent fBm ($H = 0.2$): High Mean-Reversion, "Rough" Path', fontsize=12)
axes[0].grid(alpha=0.3)

axes[1].plot(t_fbm, path_standard, color=dark_gray, lw=1.5)
axes[1].set_title('Standard Brownian Motion ($H = 0.5$): No Memory, Random Walk', fontsize=12)
axes[1].set_ylabel('Path Value', fontsize=12)
axes[1].grid(alpha=0.3)

axes[2].plot(t_fbm, path_smooth, color=navy_blue, lw=1.5)
axes[2].set_title('Persistent fBm ($H = 0.8$): Strong Momentum, "Smooth" Path', fontsize=12)
axes[2].set_xlabel('Time', fontsize=12)
axes[2].grid(alpha=0.3)

plt.suptitle('Fractional Brownian Motion: The impact of the Hurst Index ($H$)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Adjust layout to make room for suptitle
plt.show()
